# GO Enrichment Analysis — 4 Clusters (leakage-fixed model)
Uses g:Profiler REST API — no installation required beyond `requests`.

In [ ]:
import pandas as pd
import requests
import json

# Load cluster assignments from leakage-fixed simple run
df = pd.read_csv('noleak_gene_cluster_assignments.csv')
print(df['cluster'].value_counts().sort_index())
print(f'Total genes: {len(df)}')

In [ ]:
def run_gprofiler(gene_list, label):
    url = 'https://biit.cs.ut.ee/gprofiler/api/gost/profile/'
    payload = {
        'organism': 'hsapiens',
        'query': gene_list,
        'sources': ['GO:BP', 'GO:MF', 'GO:CC', 'KEGG', 'REAC'],
        'user_threshold': 0.05,
        'significance_threshold_method': 'fdr',
        'no_evidences': True
    }
    r = requests.post(url, json=payload)
    r.raise_for_status()
    results = r.json()['result']
    if not results:
        print(f'{label}: No significant terms found.')
        return pd.DataFrame()
    rows = []
    for res in results:
        rows.append({
            'source': res['source'],
            'term_id': res['native'],
            'term_name': res['name'],
            'p_value': res['p_value'],
            'intersection_size': res['intersection_size'],
            'term_size': res['term_size']
        })
    result_df = pd.DataFrame(rows).sort_values('p_value')
    print(f'{label}: {len(result_df)} significant terms')
    return result_df

In [ ]:
# Extract gene lists for all 4 clusters
cluster_genes = {}
for i in sorted(df['cluster'].unique()):
    genes = df[df['cluster'] == i]['gene'].tolist()
    cluster_genes[i] = genes
    print(f'Cluster {i}: {len(genes)} genes')

In [ ]:
# Run enrichment for all 4 clusters
results = {}
for i in sorted(cluster_genes.keys()):
    results[i] = run_gprofiler(cluster_genes[i], f'Cluster {i}')

In [ ]:
# Print top 5 terms per source per cluster
for i in sorted(results.keys()):
    df_res = results[i]
    if df_res.empty:
        continue
    print(f'\n=== Cluster {i} ({len(cluster_genes[i])} genes) ===')
    for source in ['GO:BP', 'GO:MF', 'GO:CC', 'KEGG', 'REAC']:
        sub = df_res[df_res['source'] == source].head(5)
        if len(sub) > 0:
            print(f'  {source}:')
            for _, row in sub.iterrows():
                print(f'    {row["term_name"]} (p={row["p_value"]:.2e}, n={row["intersection_size"]})')

In [ ]:
# Save per-cluster CSVs
for i in sorted(results.keys()):
    if not results[i].empty:
        results[i].to_csv(f'cluster{i}_go_enrichment_noleak.csv', index=False)

# Save compact JSON summary
summary = {}
for i in sorted(results.keys()):
    df_res = results[i]
    if df_res.empty:
        continue
    summary[f'cluster{i}'] = {}
    for source in ['GO:BP', 'GO:MF', 'GO:CC', 'KEGG', 'REAC']:
        sub = df_res[df_res['source'] == source].head(10)
        summary[f'cluster{i}'][source] = sub[['term_name', 'p_value', 'intersection_size']].to_dict('records')

with open('go_enrichment_summary_noleak.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved: cluster{0-3}_go_enrichment_noleak.csv, go_enrichment_summary_noleak.json')